# Depth Estimation Robustness Analysis

# Part 1 — Three Robustness Tests

In [1]:
from __future__ import annotations

import os

import importlib.util
import json
import math
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
#try:
#    get_ipython().run_line_magic("matplotlib", "inline")
#except NameError:
#    pass

DATA_ROOT = Path(os.environ.get("MECH5720_DATA_ROOT", r"C:/Users/user/Desktop/M2/Sensors and Signals/Major Project/major project/final_experiment_data"))
LIGHTFIELD_ROOT = DATA_ROOT / "sessions" / "lightfield"
OUTPUT_DIR = DATA_ROOT / "robustness_analysis_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = [
    DATA_ROOT / "depth_estimation_validation.py",
    DATA_ROOT / "targets.csv",
    LIGHTFIELD_ROOT / "lf_layers_5obj_01",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

spec = importlib.util.spec_from_file_location(
    "depth_base", DATA_ROOT / "depth_estimation_validation.py"
)
depth_base = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(depth_base)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

print("Data root:", DATA_ROOT)
print("Results folder:", OUTPUT_DIR)

Data root: C:\Users\user\Desktop\M2\Sensors and Signals\Major Project\major project\final_experiment_data
Results folder: C:\Users\user\Desktop\M2\Sensors and Signals\Major Project\major project\final_experiment_data\robustness_analysis_results


## 1.0 Shared Clean Calibration

In [2]:
clean_rows = []
clean_matches = {}

for session_name in depth_base.DEPTH_GROUND_TRUTH:
    result, matches = depth_base.estimate_disparity_slope(
        LIGHTFIELD_ROOT / session_name
    )
    clean_rows.append(result)
    clean_matches[session_name] = matches.copy()

clean_rows = pd.DataFrame(clean_rows)
clean_model = depth_base.fit_calibration(clean_rows)
clean_predictions = depth_base.predict_holdout(clean_rows, clean_model)
clean_metrics = depth_base.metrics(clean_predictions)

display(pd.DataFrame([clean_model | clean_metrics]))
display(clean_predictions[[
    "session", "depth_mm", "estimated_depth_mm", "error_mm"
]])

,a_px,b_px_per_mm,calibration_R2,n_holdout,RMSE_mm,MAE_mm,MAPE_percent,bias_mm,max_abs_error_mm
0,1802.494583,0.010971,0.99984,4,7.98581,6.846387,0.451765,6.846387,13.301391


,session,depth_mm,estimated_depth_mm,error_mm
4,lf_holdout_01,1300.0,1304.781510,4.781510
5,lf_holdout_02,1500.0,1507.104198,7.104198
6,lf_holdout_03,1200.0,1202.198450,2.198450
7,lf_holdout_04,1700.0,1713.301391,13.301391


## 1.1 Simulated SNR vs Depth RMSE

In [3]:
def _add_awgn(gray: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    noisy = gray.astype(np.float32) + rng.normal(0.0, sigma, gray.shape).astype(np.float32)
    return np.clip(noisy, 0, 255).astype(np.uint8)

def _fit_track(
    images: list[np.ndarray],
    positions: np.ndarray,
    roi_xyxy: tuple[int, int, int, int],
    target_snr_db: float | None = None,
    rng: np.random.Generator | None = None,
) -> dict:
    """Track one template through all views and return slope diagnostics."""
    ref_index = int(np.argmin(np.abs(positions - 100.0)))
    x1, y1, x2, y2 = roi_xyxy
    clean_grays = [cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) for image in images]

    if target_snr_db is None:
        noisy_grays = clean_grays
        noise_sigma = 0.0
    else:
        if rng is None:
            raise ValueError("rng is required when target_snr_db is provided")
        reference_signal = float(np.mean(clean_grays[ref_index][y1:y2, x1:x2]))
        noise_sigma = reference_signal / (10.0 ** (target_snr_db / 20.0))
        noisy_grays = [_add_awgn(gray, noise_sigma, rng) for gray in clean_grays]

    template = noisy_grays[ref_index][y1:y2, x1:x2]
    if template.size == 0:
        raise ValueError("Template ROI is outside the image")

    rows = []
    for view_index, (position, gray) in enumerate(zip(positions, noisy_grays)):
        response = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)
        _, score, _, location = cv2.minMaxLoc(response)
        rows.append({
            "view_index": view_index,
            "camera_x_mm": float(position),
            "object_x_px": float(location[0]),
            "object_y_px": float(location[1]),
            "match_score": float(score),
        })

    matches = pd.DataFrame(rows)
    slope, intercept = np.polyfit(matches["camera_x_mm"], matches["object_x_px"], 1)
    predicted = slope * matches["camera_x_mm"] + intercept
    residual = matches["object_x_px"] - predicted
    ss_res = float(np.sum(residual**2))
    ss_tot = float(np.sum((matches["object_x_px"] - matches["object_x_px"].mean()) ** 2))
    fit_r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else math.nan
    return {
        "abs_slope_px_per_mm": abs(float(slope)),
        "fit_R2": fit_r2,
        "min_match_score": float(matches["match_score"].min()),
        "residual_RMSE_px": float(np.sqrt(np.mean(residual**2))),
        "noise_sigma_DN": noise_sigma,
    }

HOLDOUTS = [name for name in depth_base.DEPTH_GROUND_TRUTH if "holdout" in name]
holdout_images = {
    name: depth_base.load_preview_views(LIGHTFIELD_ROOT / name)[:2]
    for name in HOLDOUTS
}

SNR_LEVELS_DB = [15, 20, 25, 30, 35, 40]
N_NOISE_REALISATIONS = 20
MATCH_SCORE_THRESHOLD = 0.60
FIT_R2_THRESHOLD = 0.98

noise_prediction_rows = []
noise_trial_rows = []

for snr_db in SNR_LEVELS_DB:
    for trial in range(N_NOISE_REALISATIONS):
        trial_errors = []
        flagged = 0
        for holdout_index, session_name in enumerate(HOLDOUTS):
            images, positions = holdout_images[session_name]
            rng = np.random.default_rng(10_000 * snr_db + 100 * trial + holdout_index)
            diagnostic = _fit_track(
                images,
                positions,
                depth_base.TEMPLATE_ROI_XYXY,
                target_snr_db=snr_db,
                rng=rng,
            )
            slope_abs = diagnostic["abs_slope_px_per_mm"]
            estimated_depth = clean_model["a_px"] / (
                slope_abs - clean_model["b_px_per_mm"]
            )
            true_depth = float(depth_base.DEPTH_GROUND_TRUTH[session_name])
            error = float(estimated_depth - true_depth)
            is_flagged = (
                diagnostic["min_match_score"] < MATCH_SCORE_THRESHOLD
                or diagnostic["fit_R2"] < FIT_R2_THRESHOLD
            )
            flagged += int(is_flagged)
            trial_errors.append(error)
            noise_prediction_rows.append({
                "target_snr_db": snr_db,
                "trial": trial,
                "session": session_name,
                "depth_mm": true_depth,
                "estimated_depth_mm": estimated_depth,
                "error_mm": error,
                "flagged_match": is_flagged,
                **diagnostic,
            })

        trial_errors = np.asarray(trial_errors, dtype=float)
        noise_trial_rows.append({
            "target_snr_db": snr_db,
            "trial": trial,
            "RMSE_mm": float(np.sqrt(np.mean(trial_errors**2))),
            "MAE_mm": float(np.mean(np.abs(trial_errors))),
            "flagged_fraction": flagged / len(HOLDOUTS),
        })

noise_predictions = pd.DataFrame(noise_prediction_rows)
noise_trials = pd.DataFrame(noise_trial_rows)

noise_summary = (
    noise_trials.groupby("target_snr_db")
    .agg(
        RMSE_mean_mm=("RMSE_mm", "mean"),
        RMSE_SD_mm=("RMSE_mm", "std"),
        RMSE_CI95_low=("RMSE_mm", lambda x: np.quantile(x, 0.025)),
        RMSE_CI95_high=("RMSE_mm", lambda x: np.quantile(x, 0.975)),
        MAE_mean_mm=("MAE_mm", "mean"),
        flagged_fraction=("flagged_fraction", "mean"),
    )
    .reset_index()
)

noise_predictions.to_csv(OUTPUT_DIR / "noise_depth_predictions.csv", index=False)
noise_trials.to_csv(OUTPUT_DIR / "noise_depth_trials.csv", index=False)
noise_summary.to_csv(OUTPUT_DIR / "noise_depth_summary.csv", index=False)

display(noise_summary)

,target_snr_db,RMSE_mean_mm,RMSE_SD_mm,RMSE_CI95_low,RMSE_CI95_high,MAE_mean_mm,flagged_fraction
0,15,8.521408,2.713582,5.025711,12.951684,7.039742,1.00
1,20,8.753996,1.352316,6.466631,10.937449,7.119676,0.25
2,25,8.511664,0.630968,7.381225,9.040665,7.146065,0.00
3,30,8.034400,0.237575,7.942207,8.539609,6.872503,0.00
4,35,7.985810,0.000000,7.985810,7.985810,6.846387,0.00
5,40,7.985810,0.000000,7.985810,7.985810,6.846387,0.00


In [4]:
fig, ax1 = plt.subplots(figsize=(8.5, 5.2))

x = noise_summary["target_snr_db"].to_numpy(float)
y = noise_summary["RMSE_mean_mm"].to_numpy(float)
low = noise_summary["RMSE_CI95_low"].to_numpy(float)
high = noise_summary["RMSE_CI95_high"].to_numpy(float)

ax1.plot(x, y, marker="o", color="#087f8c", label="Simulated-noise RMSE")
ax1.fill_between(x, low, high, color="#087f8c", alpha=0.18, label="95% Monte Carlo interval")
ax1.axhline(clean_metrics["RMSE_mm"], color="#333333", linestyle="--", label="Clean holdout RMSE")
ax1.set_xlabel("Target simulated SNR (dB)")
ax1.set_ylabel("Holdout depth RMSE (mm)")
ax1.set_title("Depth-estimation robustness to simulated image noise")

ax2 = ax1.twinx()
ax2.plot(
    x,
    100 * noise_summary["flagged_fraction"].to_numpy(float),
    marker="s",
    color="#d95f02",
    label="Flagged matches",
)
ax2.set_ylabel("Flagged match fraction (%)")
ax2.set_ylim(bottom=0)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="best")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_depth_rmse_vs_simulated_snr.png", dpi=240)
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_65700\1567568233.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.2 Camera-Position Uncertainty

In [5]:
def _slope_from_matches_with_position_error(
    matches: pd.DataFrame,
    position_sigma_mm: float,
    rng: np.random.Generator,
) -> float:
    x_used = matches["camera_x_mm"].to_numpy(float) + rng.normal(
        0.0, position_sigma_mm, len(matches)
    )
    object_x = matches["object_x_px"].to_numpy(float)
    slope, _ = np.polyfit(x_used, object_x, 1)
    return abs(float(slope))

POSITION_SIGMAS_MM = [0.0, 0.25, 0.5, 1.0, 2.0]
N_POSITION_TRIALS = 500
position_trial_rows = []

for sigma_mm in POSITION_SIGMAS_MM:
    for trial in range(N_POSITION_TRIALS):
        rng = np.random.default_rng(500_000 + int(1000 * sigma_mm) + trial)
        rows = []
        for session_name, true_depth in depth_base.DEPTH_GROUND_TRUTH.items():
            abs_slope = _slope_from_matches_with_position_error(
                clean_matches[session_name], sigma_mm, rng
            )
            rows.append({
                "session": session_name,
                "split": "calibration" if "train" in session_name else "holdout",
                "depth_mm": float(true_depth),
                "abs_slope_px_per_mm": abs_slope,
            })

        trial_df = pd.DataFrame(rows)
        trial_model = depth_base.fit_calibration(trial_df)
        predictions = depth_base.predict_holdout(trial_df, trial_model)
        trial_metrics = depth_base.metrics(predictions)
        position_trial_rows.append({
            "position_sigma_mm": sigma_mm,
            "trial": trial,
            "calibration_R2": trial_model["calibration_R2"],
            **trial_metrics,
        })

position_trials = pd.DataFrame(position_trial_rows)
position_summary = (
    position_trials.groupby("position_sigma_mm")
    .agg(
        RMSE_mean_mm=("RMSE_mm", "mean"),
        RMSE_SD_mm=("RMSE_mm", "std"),
        RMSE_CI95_low=("RMSE_mm", lambda x: np.quantile(x, 0.025)),
        RMSE_CI95_high=("RMSE_mm", lambda x: np.quantile(x, 0.975)),
        MAE_mean_mm=("MAE_mm", "mean"),
        calibration_R2_mean=("calibration_R2", "mean"),
    )
    .reset_index()
)

position_trials.to_csv(OUTPUT_DIR / "position_uncertainty_trials.csv", index=False)
position_summary.to_csv(OUTPUT_DIR / "position_uncertainty_summary.csv", index=False)

display(position_summary)

,position_sigma_mm,RMSE_mean_mm,RMSE_SD_mm,RMSE_CI95_low,RMSE_CI95_high,MAE_mean_mm,calibration_R2_mean
0,0.00,7.985810,0.000000,7.985810,7.985810,6.846387,0.999840
1,0.25,8.252891,1.520617,5.223822,11.294039,6.968091,0.999822
2,0.50,8.955740,2.737020,3.844756,14.725073,7.485650,0.999736
3,1.00,11.500498,4.519391,4.103371,20.782976,9.655462,0.999481
4,2.00,19.040573,7.788865,6.828702,36.336146,16.037293,0.998352


In [6]:
fig, ax = plt.subplots(figsize=(8.2, 5.0))
x = position_summary["position_sigma_mm"].to_numpy(float)
y = position_summary["RMSE_mean_mm"].to_numpy(float)
low = position_summary["RMSE_CI95_low"].to_numpy(float)
high = position_summary["RMSE_CI95_high"].to_numpy(float)

ax.plot(x, y, marker="o", color="#087f8c")
ax.fill_between(x, low, high, color="#087f8c", alpha=0.18)
ax.axhline(clean_metrics["RMSE_mm"], color="#333333", linestyle="--", label="No added coordinate error")
ax.set_xlabel("Added camera-position uncertainty, σ (mm)")
ax.set_ylabel("Holdout depth RMSE (mm)")
ax.set_title("Sensitivity to camera-position coordinate uncertainty")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_depth_rmse_vs_position_uncertainty.png", dpi=240)
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_65700\3043598463.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3 Cross-Object Generalisation (Five-Object Scene)

In [7]:
FIVE_OBJECT_ROIS = {
    "FIVE_LEFT_01": (123, 545, 160, 730),
    "FIVE_LEFT_02": (178, 545, 219, 731),
    "FIVE_LEFT_03": (236, 630, 305, 731),
    "FIVE_LEFT_04": (365, 692, 402, 770),
    "FIVE_LEFT_05": (434, 520, 505, 780),
}

five_images, five_positions, five_paths = depth_base.load_preview_views(
    LIGHTFIELD_ROOT / "lf_layers_5obj_01"
)
five_ref_index = int(np.argmin(np.abs(five_positions - 100.0)))
five_reference = five_images[five_ref_index]

fig, ax = plt.subplots(figsize=(6, 9))
ax.imshow(five_reference)
for target_id, (x1, y1, x2, y2) in FIVE_OBJECT_ROIS.items():
    rectangle = plt.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        fill=False, linewidth=2, edgecolor="#ffcc00"
    )
    ax.add_patch(rectangle)
    ax.text(
        x1, y1 - 8, target_id.replace("FIVE_LEFT_", "Object "),
        color="black", fontsize=9,
        bbox={"facecolor": "#ffcc00", "alpha": 0.85, "pad": 2},
    )
ax.set_title("Pre-declared ROIs in the five-object centre view")
ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_five_object_rois.png", dpi=220)
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_65700\3841982675.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
targets = pd.read_csv(DATA_ROOT / "targets.csv").set_index("target_id")
five_object_rows = []

for target_id, roi in FIVE_OBJECT_ROIS.items():
    diagnostic = _fit_track(
        five_images,
        five_positions,
        roi,
        target_snr_db=None,
    )
    abs_slope = diagnostic["abs_slope_px_per_mm"]
    estimated_depth = clean_model["a_px"] / (
        abs_slope - clean_model["b_px_per_mm"]
    )
    true_depth = float(targets.loc[target_id, "depth_mm"])
    error = float(estimated_depth - true_depth)
    five_object_rows.append({
        "target_id": target_id,
        "description": targets.loc[target_id, "description"],
        "depth_mm": true_depth,
        "estimated_depth_mm": estimated_depth,
        "error_mm": error,
        "abs_error_mm": abs(error),
        "percentage_error": 100.0 * abs(error) / true_depth,
        **diagnostic,
    })

five_object_predictions = pd.DataFrame(five_object_rows)
five_object_rmse = float(np.sqrt(np.mean(five_object_predictions["error_mm"] ** 2)))
five_object_mae = float(np.mean(five_object_predictions["abs_error_mm"]))
five_object_mape = float(np.mean(five_object_predictions["percentage_error"]))

five_object_summary = pd.DataFrame([{
    "n_objects": len(five_object_predictions),
    "RMSE_mm": five_object_rmse,
    "MAE_mm": five_object_mae,
    "MAPE_percent": five_object_mape,
    "min_match_score": float(five_object_predictions["min_match_score"].min()),
    "min_disparity_fit_R2": float(five_object_predictions["fit_R2"].min()),
}])

five_object_predictions.to_csv(OUTPUT_DIR / "five_object_predictions.csv", index=False)
five_object_summary.to_csv(OUTPUT_DIR / "five_object_summary.csv", index=False)

display(five_object_predictions[[
    "target_id", "depth_mm", "estimated_depth_mm", "error_mm",
    "fit_R2", "min_match_score"
]])
display(five_object_summary)

,target_id,depth_mm,estimated_depth_mm,error_mm,fit_R2,min_match_score
0,FIVE_LEFT_01,1400.0,1406.601968,6.601968,0.999924,0.884934
1,FIVE_LEFT_02,1500.0,1509.398818,9.398818,0.999933,0.929662
2,FIVE_LEFT_03,1700.0,1698.138547,-1.861453,0.999952,0.944090
3,FIVE_LEFT_04,1200.0,1209.286314,9.286314,0.999919,0.949623
4,FIVE_LEFT_05,1100.0,1112.485428,12.485428,0.999900,0.941965


,n_objects,RMSE_mm,MAE_mm,MAPE_percent,min_match_score,min_disparity_fit_R2
0,5,8.689192,7.926796,0.623311,0.884934,0.9999


In [9]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
minimum = min(
    five_object_predictions["depth_mm"].min(),
    five_object_predictions["estimated_depth_mm"].min(),
) - 40
maximum = max(
    five_object_predictions["depth_mm"].max(),
    five_object_predictions["estimated_depth_mm"].max(),
) + 40

ax.plot([minimum, maximum], [minimum, maximum], linestyle="--", color="#555555", label="Ideal")
ax.scatter(
    five_object_predictions["depth_mm"],
    five_object_predictions["estimated_depth_mm"],
    s=65, color="#087f8c", label="Five-object scene"
)
for _, row in five_object_predictions.iterrows():
    ax.annotate(
        row["target_id"].replace("FIVE_LEFT_", ""),
        (row["depth_mm"], row["estimated_depth_mm"]),
        xytext=(5, 5), textcoords="offset points"
    )
ax.set_xlim(minimum, maximum)
ax.set_ylim(minimum, maximum)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Measured depth (mm)")
ax.set_ylabel("Estimated depth (mm)")
ax.set_title(f"Cross-object generalisation: RMSE = {five_object_rmse:.2f} mm")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_five_object_generalisation.png", dpi=240)
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_65700\3776432893.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
final_summary = {
    "clean_holdout_RMSE_mm": clean_metrics["RMSE_mm"],
    "simulated_noise": noise_summary.to_dict(orient="records"),
    "position_uncertainty": position_summary.to_dict(orient="records"),
    "five_object_generalisation": five_object_summary.iloc[0].to_dict(),
}

with open(OUTPUT_DIR / "robustness_summary.json", "w", encoding="utf-8") as handle:
    json.dump(final_summary, handle, indent=2)

print("Saved outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

Saved outputs:
 - 01_depth_rmse_vs_simulated_snr.png
 - 02_depth_rmse_vs_position_uncertainty.png
 - 03_five_object_rois.png
 - 04_five_object_generalisation.png
 - five_object_predictions.csv
 - five_object_summary.csv
 - noise_depth_predictions.csv
 - noise_depth_summary.csv
 - noise_depth_trials.csv
 - position_uncertainty_summary.csv
 - position_uncertainty_trials.csv
 - robustness_summary.json


# Part 2 — RAW-Domain SNR vs Depth RMSE

In [11]:
import os, json, math, itertools, importlib.util
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import tifffile

DECODER = os.environ.get("MECH5720_DECODER", "builtin")
if DECODER == "rawpy":
    import rawpy

DATA_ROOT = Path(os.environ.get(
    "MECH5720_DATA_ROOT",
    r"C:/Users/user/Desktop/M2/Sensors and Signals/Major Project/major project/final_experiment_data"))
LIGHTFIELD_ROOT = DATA_ROOT / "sessions" / "lightfield"
SNR_ROOT = DATA_ROOT / "sessions" / "snr"
OUTPUT_DIR = DATA_ROOT / "raw_snr_depth_robustness_results"
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

spec = importlib.util.spec_from_file_location("depth_base", DATA_ROOT / "depth_estimation_validation.py")
depth_base = importlib.util.module_from_spec(spec)
spec.loader.exec_module(depth_base)

BASE_EXPOSURE_MS = 32.0
SIM_EXPOSURES_MS = [32.0, 24.0, 16.0]
EXTRAPOLATE = False
EXTRAPOLATED_EXPOSURES_MS = [8.0, 4.0]
N_TRIALS = 20
RNG_SEED = 20261103
PREVIEW_WIDTH_PX = 675

FIGS = {}
TEAL, ORANGE, GREY = "#0B7A85", "#C2570C", "#5F6670"

CAL_SESSIONS = [s for s in depth_base.DEPTH_GROUND_TRUTH if "train" in s]
HOLDOUT_SESSIONS = [s for s in depth_base.DEPTH_GROUND_TRUTH if "holdout" in s]
print("Decoder:", DECODER)
print("Data root:", DATA_ROOT)
print("Calibration sessions:", CAL_SESSIONS)
print("Holdout sessions:", HOLDOUT_SESSIONS)
print("Outputs:", OUTPUT_DIR)

Decoder: builtin
Data root: C:\Users\user\Desktop\M2\Sensors and Signals\Major Project\major project\final_experiment_data
Calibration sessions: ['lf_train_01', 'lf_train_02', 'lf_train_03', 'lf_train_04']
Holdout sessions: ['lf_holdout_01', 'lf_holdout_02', 'lf_holdout_03', 'lf_holdout_04']
Outputs: C:\Users\user\Desktop\M2\Sensors and Signals\Major Project\major project\final_experiment_data\raw_snr_depth_robustness_results


## 2.1 RAW Decoding and Geometry

In [12]:
LUMA = np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)

XYZ_RGB = np.array([[0.412453, 0.357580, 0.180423],
                    [0.212671, 0.715160, 0.072169],
                    [0.019334, 0.119193, 0.950227]])

def _rational(v):
    v = np.asarray(v, dtype=float).ravel()
    return v[0::2] / v[1::2]

def read_dng_cfa(path: Path):

    with tifffile.TiffFile(path) as tif:
        page = tif.pages[0]
        h, w = page.shape
        fh = tif.filehandle
        chunks = []
        for off, n in zip(page.dataoffsets, page.databytecounts):
            fh.seek(off); chunks.append(fh.read(n))
        tags = {t.name: t.value for t in page.tags.values()}
    bps = int(tags.get("BitsPerSample", 10))
    if bps != 10:
        raise ValueError(f"{path.name}: expected 10-bit DNG, got {bps}")
    b = np.frombuffer(b"".join(chunks), np.uint8)[: h * w * 10 // 8].reshape(-1, 5).astype(np.uint16)
    px = np.stack([(b[:, 0] << 2) | (b[:, 1] >> 6), ((b[:, 1] & 0x3F) << 4) | (b[:, 2] >> 4),
                   ((b[:, 2] & 0x0F) << 6) | (b[:, 3] >> 2), ((b[:, 3] & 0x03) << 8) | b[:, 4]], axis=1)
    return px.reshape(h, w), tags

def builtin_rgb16(path: Path) -> np.ndarray:
    cfa, tags = read_dng_cfa(path)
    black = float(np.mean(tags["BlackLevel"])); white = float(tags["WhiteLevel"])
    if bytes(tags["CFAPattern"]) != b"\x01\x02\x00\x01":
        raise ValueError(f"{path.name}: unexpected CFA pattern {tags['CFAPattern']}")
    rgb = cv2.cvtColor(cfa, cv2.COLOR_BayerGR2RGB).astype(np.float32)
    rgb = (rgb - black) / (white - black)
    rgb *= (1.0 / _rational(tags["AsShotNeutral"])).astype(np.float32)
    cam_xyz = _rational(tags["ColorMatrix1"]).reshape(3, 3)
    cam_rgb = cam_xyz @ XYZ_RGB
    cam_rgb /= cam_rgb.sum(axis=1, keepdims=True)
    rgb_cam = np.linalg.inv(cam_rgb).astype(np.float32)
    return (rgb @ rgb_cam.T) * 65535.0

def load_luminance(path: Path) -> np.ndarray:

    if DECODER == "rawpy":
        with rawpy.imread(str(path)) as raw:
            rgb = raw.postprocess(use_camera_wb=True, no_auto_bright=True, output_bps=16, gamma=(1, 1)).astype(np.float32)
    else:
        rgb = builtin_rgb16(path)
    rgb = np.rot90(rgb, k=3)
    return np.ascontiguousarray(rgb @ LUMA)

def read_manifest(session_dir: Path) -> pd.DataFrame:
    return pd.read_csv(session_dir / "manifest.csv")

first_manifest = read_manifest(LIGHTFIELD_ROOT / CAL_SESSIONS[0])
first_img = load_luminance(LIGHTFIELD_ROOT / CAL_SESSIONS[0] / first_manifest["filename"].iloc[0])
IMG_H, IMG_W = first_img.shape
SCALE = IMG_W / PREVIEW_WIDTH_PX
px1, py1, px2, py2 = depth_base.TEMPLATE_ROI_XYXY
ROI_FULL = tuple(int(round(v * SCALE)) for v in (px1, py1, px2, py2))
BAND_Y0 = max(ROI_FULL[1] - 160, 0)
BAND_Y1 = min(ROI_FULL[3] + 160, IMG_H)
ROI_BAND = (ROI_FULL[0], ROI_FULL[1] - BAND_Y0, ROI_FULL[2], ROI_FULL[3] - BAND_Y0)

print(f"RAW luminance image: {IMG_W} x {IMG_H} px (W x H), scale from preview = {SCALE:.3f}")
print("Target ROI (full frame, x1 y1 x2 y2):", ROI_FULL)
print("Stored band rows:", (BAND_Y0, BAND_Y1))

RAW luminance image: 1080 x 1920 px (W x H), scale from preview = 1.600
Target ROI (full frame, x1 y1 x2 y2): (376, 856, 624, 1152)
Stored band rows: (696, 1312)


## 2.2 Load Light-Field Sessions

In [13]:
def load_lightfield_session(name: str):
    man = read_manifest(LIGHTFIELD_ROOT / name)
    repeats = sorted(man["repeat_index"].unique())
    xs = np.sort(man["nominal_x_mm"].unique()).astype(float)
    stack = np.empty((len(repeats), len(xs), BAND_Y1 - BAND_Y0, IMG_W), dtype=np.float32)
    for i, rep in enumerate(repeats):
        sel = man[man["repeat_index"] == rep].sort_values("nominal_x_mm")
        if len(sel) != len(xs):
            raise ValueError(f"{name} repeat {rep}: expected {len(xs)} views, found {len(sel)}")
        for j, fname in enumerate(sel["filename"]):
            stack[i, j] = load_luminance(LIGHTFIELD_ROOT / name / fname)[BAND_Y0:BAND_Y1]
    return stack, xs

LF = {}
for name in CAL_SESSIONS + HOLDOUT_SESSIONS:
    LF[name] = load_lightfield_session(name)
    print(f"{name}: stack {LF[name][0].shape}, positions {LF[name][1].min():.0f}-{LF[name][1].max():.0f} mm")
XS = LF[CAL_SESSIONS[0]][1]
REF_VIEW = int(np.argmin(np.abs(XS - 100.0)))

lf_train_01: stack (3, 10, 616, 1080), positions 0-180 mm
lf_train_02: stack (3, 10, 616, 1080), positions 0-180 mm
lf_train_03: stack (3, 10, 616, 1080), positions 0-180 mm
lf_train_04: stack (3, 10, 616, 1080), positions 0-180 mm
lf_holdout_01: stack (3, 10, 616, 1080), positions 0-180 mm
lf_holdout_02: stack (3, 10, 616, 1080), positions 0-180 mm
lf_holdout_03: stack (3, 10, 616, 1080), positions 0-180 mm
lf_holdout_04: stack (3, 10, 616, 1080), positions 0-180 mm


## 2.3 Dark Level and SNR

In [14]:
def load_stack_full(session_dir: Path) -> tuple[np.ndarray, pd.DataFrame]:
    man = read_manifest(session_dir).sort_values("repeat_index")
    frames = np.stack([load_luminance(session_dir / f) for f in man["filename"]])
    return frames, man

dark_frames, _ = load_stack_full(SNR_ROOT / "snr_dark")
x1, y1, x2, y2 = ROI_FULL
DARK_LEVEL = float(dark_frames[:, y1:y2, x1:x2].mean())
print(f"Dark level in target ROI: {DARK_LEVEL:.2f} (16-bit linear luminance units)")

def snr_from_repeats(stack: np.ndarray, roi, dark: float = None) -> dict:

    dark = DARK_LEVEL if dark is None else dark
    a, b, c, d = roi
    r = stack[:, b:d, a:c]
    signal = float(r.mean(axis=0).mean() - dark)
    noise = float(r.std(axis=0, ddof=1).mean())
    return {"signal": signal, "noise": noise, "SNR_dB": 20 * math.log10(signal / noise)}

baseline_snr = pd.DataFrame([
    {"session": n, **snr_from_repeats(LF[n][0][:, REF_VIEW], ROI_BAND)} for n in CAL_SESSIONS + HOLDOUT_SESSIONS])
display(baseline_snr.round(3))
print(f"Mean single-view SNR at 32 ms (target ROI): {baseline_snr['SNR_dB'].mean():.2f} dB "
      f"(slide 6 reports 27.09 dB for lf_train_01 with a slightly different ROI)")

Dark level in target ROI: -62.53 (16-bit linear luminance units)


,session,signal,noise,SNR_dB
0,lf_train_01,31468.268,1160.176,28.667
1,lf_train_02,29248.188,1076.146,28.685
2,lf_train_03,31160.449,1118.048,28.903
3,lf_train_04,31996.809,1146.859,28.912
4,lf_holdout_01,31351.072,1116.714,28.966
5,lf_holdout_02,34602.926,1193.172,29.248
6,lf_holdout_03,28166.928,1066.917,28.432
7,lf_holdout_04,31935.104,1126.817,29.048


Mean single-view SNR at 32 ms (target ROI): 28.86 dB (slide 6 reports 27.09 dB for lf_train_01 with a slightly different ROI)


## 2.4 Noise Model: Photon Transfer Curve

In [15]:
cy, cx = IMG_H // 2, IMG_W // 2
CENTRE = (cx - 150, cy - 250, cx + 150, cy + 250)

def centre_stats(frames: np.ndarray):
    a, b, c, d = CENTRE
    r = frames[:, b:d, a:c]
    return r.mean(axis=0).ravel() - DARK_LEVEL, r.var(axis=0, ddof=1).ravel()

ptc_rows, S_all, V_all, snr_frames = [], [], [], {}
sa, sv = centre_stats(dark_frames); S_all.append(sa); V_all.append(sv)
for sess in ["snr_uniform_low", "snr_uniform_mid", "snr_uniform_high"]:
    frames, man = load_stack_full(SNR_ROOT / sess)
    exp_ms = float(man["exposure_us_requested"].iloc[0]) / 1000.0
    snr_frames[exp_ms] = frames
    s, v = centre_stats(frames); S_all.append(s); V_all.append(v)
    m = snr_from_repeats(frames, CENTRE)
    ptc_rows.append({"session": sess, "exposure_ms": exp_ms, **m})
measured_snr = pd.DataFrame(ptc_rows).sort_values("exposure_ms").reset_index(drop=True)

S_all, V_all = np.concatenate(S_all), np.concatenate(V_all)
edges = np.quantile(S_all, np.linspace(0, 1, 61))
idx = np.clip(np.digitize(S_all, edges) - 1, 0, 59)
bin_S = np.array([S_all[idx == k].mean() for k in range(60) if np.any(idx == k)])
bin_V = np.array([V_all[idx == k].mean() for k in range(60) if np.any(idx == k)])
NOISE_A, NOISE_B = np.polyfit(bin_S, bin_V, 1)
NOISE_B = max(NOISE_B, 1e-6)

def model_var(signal):
    return NOISE_A * np.clip(signal, 0, None) + NOISE_B

measured_snr["model_SNR_dB"] = 20 * np.log10(measured_snr["signal"] / np.sqrt(model_var(measured_snr["signal"])))
measured_snr["signal_ratio_vs_32ms"] = measured_snr["signal"] / measured_snr.loc[measured_snr["exposure_ms"].idxmax(), "signal"]
print(f"Noise model: variance = {NOISE_A:.4f} * signal + {NOISE_B:.2f}")
display(measured_snr.round(3))

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.scatter(bin_S, bin_V, s=18, color=GREY, label="binned pixels (dark + 16/24/32 ms)")
xx = np.linspace(0, bin_S.max() * 1.05, 100)
ax.plot(xx, model_var(xx), color=TEAL, lw=2, label=f"fit: var = {NOISE_A:.3f}·S + {NOISE_B:.1f}")
ax.set(xlabel="Dark-corrected signal S (linear luminance units)", ylabel="Temporal variance",
       title="Photon transfer curve from measured RAW frames")
ax.legend(frameon=False); ax.grid(alpha=0.3)
FIGS["01_photon_transfer_curve"] = fig

Noise model: variance = 61.1611 * signal + 21874.31


,session,exposure_ms,signal,noise,SNR_dB,model_SNR_dB,signal_ratio_vs_32ms
0,snr_uniform_low,16.0,18529.139,1023.437,25.156,24.731,0.50
1,snr_uniform_mid,24.0,27774.008,1242.789,26.985,26.516,0.75
2,snr_uniform_high,32.0,37040.273,1426.543,28.288,27.780,1.00


## 2.5 Noise Model Validation

In [16]:
rng = np.random.default_rng(RNG_SEED)

hi_frames = snr_frames[max(snr_frames)]
avg_rows = []
for m in (1, 2, 3):
    groups = hi_frames[: (len(hi_frames) // m) * m].reshape(-1, m, *hi_frames.shape[1:]).mean(axis=1)
    r = snr_from_repeats(groups, CENTRE)
    avg_rows.append({"frames_averaged": m, "measured_noise": r["noise"], "measured_SNR_dB": r["SNR_dB"]})
avg_check = pd.DataFrame(avg_rows)
avg_check["sqrtN_predicted_noise"] = avg_check.loc[0, "measured_noise"] / np.sqrt(avg_check["frames_averaged"])
avg_check["SNR_gain_dB_measured"] = avg_check["measured_SNR_dB"] - avg_check.loc[0, "measured_SNR_dB"]
avg_check["SNR_gain_dB_theory"] = 10 * np.log10(avg_check["frames_averaged"])
display(avg_check.round(3))

base32 = hi_frames.mean(axis=0)
S32 = base32 - DARK_LEVEL
var_base32 = model_var(S32) / len(hi_frames)
val_rows = []
for exp_ms in sorted(snr_frames):
    k = exp_ms / max(snr_frames)
    add_sd = np.sqrt(np.clip(model_var(k * S32) - k**2 * var_base32, 0, None))
    sims = np.stack([DARK_LEVEL + k * S32 + rng.standard_normal(S32.shape).astype(np.float32) * add_sd
                     for _ in range(len(hi_frames))])
    sim = snr_from_repeats(sims, CENTRE)
    meas = measured_snr.loc[measured_snr["exposure_ms"] == exp_ms].iloc[0]
    val_rows.append({"exposure_ms": exp_ms, "measured_SNR_dB": meas["SNR_dB"],
                     "simulated_SNR_dB": sim["SNR_dB"], "difference_dB": sim["SNR_dB"] - meas["SNR_dB"]})
model_validation = pd.DataFrame(val_rows)
display(model_validation.round(3))

fig, ax = plt.subplots(figsize=(6.0, 4.2))
ax.plot(model_validation["exposure_ms"], model_validation["measured_SNR_dB"], "o-", color=TEAL, ms=8, label="measured (24 frames)")
ax.plot(model_validation["exposure_ms"], model_validation["simulated_SNR_dB"], "s--", color=ORANGE, ms=7, label="simulated from 32 ms")
ax.set(xlabel="Exposure time (ms)", ylabel="SNR (dB)", title="Noise-model validation on the uniform SNR target")
ax.set_xticks(model_validation["exposure_ms"]); ax.legend(frameon=False); ax.grid(alpha=0.3)
FIGS["02_noise_model_validation"] = fig

,frames_averaged,measured_noise,measured_SNR_dB,sqrtN_predicted_noise,SNR_gain_dB_measured,SNR_gain_dB_theory
0,1,1426.543,28.288,1426.543,0.000,0.000
1,2,997.193,31.398,1008.718,3.110,3.010
2,3,804.894,33.259,823.615,4.971,4.771


,exposure_ms,measured_SNR_dB,simulated_SNR_dB,difference_dB
0,16.0,25.156,24.912,-0.244
1,24.0,26.985,26.751,-0.234
2,32.0,28.288,28.060,-0.228


## 2.6 Depth Estimation and Calibration (RAW)

In [17]:
def track_slope(views: np.ndarray, xs: np.ndarray, roi=ROI_BAND) -> dict:
    a, b, c, d = roi
    tpl = np.ascontiguousarray(views[REF_VIEW][b:d, a:c])
    us, scores = [], []
    for v in views:
        resp = cv2.matchTemplate(np.ascontiguousarray(v), tpl, cv2.TM_CCOEFF_NORMED)
        _, score, _, (xi, yi) = cv2.minMaxLoc(resp)
        dx = depth_base.quadratic_peak_offset(resp[yi, xi - 1], resp[yi, xi], resp[yi, xi + 1]) if 0 < xi < resp.shape[1] - 1 else 0.0
        us.append(xi + dx); scores.append(score)
    us = np.asarray(us)
    slope, icpt = np.polyfit(xs, us, 1)
    res = us - (slope * xs + icpt)
    return {"abs_slope": abs(float(slope)), "fit_R2": 1 - np.sum(res**2) / np.sum((us - us.mean())**2),
            "min_match_score": float(min(scores))}

cal_rows = []
for n in CAL_SESSIONS:
    stack, xs = LF[n]
    cal_rows.append({"session": n, "depth_mm": depth_base.DEPTH_GROUND_TRUTH[n], **track_slope(stack.mean(axis=0), xs)})
cal_df = pd.DataFrame(cal_rows)
CAL_A, CAL_B = np.polyfit(1 / cal_df["depth_mm"], cal_df["abs_slope"], 1)
pred = CAL_A / cal_df["depth_mm"] + CAL_B
CAL_R2 = 1 - np.sum((cal_df["abs_slope"] - pred)**2) / np.sum((cal_df["abs_slope"] - cal_df["abs_slope"].mean())**2)

def slope_to_depth(abs_slope):
    return CAL_A / (abs_slope - CAL_B)

display(cal_df.round(5))
print(f"RAW calibration: |s| = {CAL_A:.2f} / Z + {CAL_B:.5f}   (R^2 = {CAL_R2:.6f}, full-resolution px/mm)")

,session,depth_mm,abs_slope,fit_R2,min_match_score
0,lf_train_01,1810.0,1.61784,0.99986,0.97300
1,lf_train_02,1100.0,2.64659,0.99998,0.95821
2,lf_train_03,1400.0,2.06683,0.99996,0.97332
3,lf_train_04,1600.0,1.81195,0.99993,0.96955


RAW calibration: |s| = 2897.27 / Z + 0.00709   (R^2 = 0.999562, full-resolution px/mm)


## 2.7 Experiment A: Real Multi-Frame Averaging

In [18]:
snr1_by_session = baseline_snr.set_index("session")["SNR_dB"]
avg_pred_rows = []
for n in HOLDOUT_SESSIONS:
    stack, xs = LF[n]
    z_true = depth_base.DEPTH_GROUND_TRUTH[n]
    for m in (1, 2, 3):
        for combo in itertools.combinations(range(stack.shape[0]), m):
            t = track_slope(stack[list(combo)].mean(axis=0), xs)
            z = slope_to_depth(t["abs_slope"])
            avg_pred_rows.append({"session": n, "depth_mm": z_true, "frames_averaged": m, "frames": combo,
                                  "SNR_dB": snr1_by_session[n] + 10 * math.log10(m),
                                  "estimated_depth_mm": z, "error_mm": z - z_true, **t})
avg_pred = pd.DataFrame(avg_pred_rows)

def summarise(df, by):
    g = df.groupby(by)
    return pd.DataFrame({
        "SNR_dB": g["SNR_dB"].mean(),
        "n_predictions": g.size(),
        "RMSE_mm": g["error_mm"].apply(lambda e: float(np.sqrt(np.mean(e**2)))),
        "MAE_mm": g["error_mm"].apply(lambda e: float(np.mean(np.abs(e)))),
        "bias_mm": g["error_mm"].mean(),
        "min_match_score": g["min_match_score"].min(),
    }).reset_index()

avg_summary = summarise(avg_pred, "frames_averaged")
display(avg_summary.round(3))

,frames_averaged,SNR_dB,n_predictions,RMSE_mm,MAE_mm,bias_mm,min_match_score
0,1,28.924,12,7.245,6.308,6.308,0.942
1,2,31.934,12,7.196,6.229,6.229,0.963
2,3,33.695,4,7.228,6.261,6.261,0.968


## 2.8 Experiment B: Calibrated Noise Injection

In [ ]:
exposures = list(SIM_EXPOSURES_MS) + (list(EXTRAPOLATED_EXPOSURES_MS) if EXTRAPOLATE else [])
sim_pred_rows, sim_trial_rows = [], []
rng = np.random.default_rng(RNG_SEED + 1)

for n in HOLDOUT_SESSIONS:
    stack, xs = LF[n]
    z_true = depth_base.DEPTH_GROUND_TRUTH[n]
    S = stack.mean(axis=0) - DARK_LEVEL
    var_base = model_var(S) / stack.shape[0]
    for exp_ms in exposures:
        k = exp_ms / BASE_EXPOSURE_MS
        add_sd = np.sqrt(np.clip(model_var(k * S) - k**2 * var_base, 0, None)).astype(np.float32)
        for trial in range(N_TRIALS):
            reps = [DARK_LEVEL + k * S + rng.standard_normal(S.shape, dtype=np.float32) * add_sd
                    for _ in range(stack.shape[0])]
            snr = snr_from_repeats(np.stack([r[REF_VIEW] for r in reps]), ROI_BAND)
            for rep_i, views in enumerate(reps):
                t = track_slope(views, xs)
                z = slope_to_depth(t["abs_slope"])
                sim_pred_rows.append({"session": n, "depth_mm": z_true, "exposure_ms": exp_ms,
                                      "extrapolated": exp_ms < min(SIM_EXPOSURES_MS), "trial": trial, "repeat": rep_i,
                                      "SNR_dB": snr["SNR_dB"], "estimated_depth_mm": z, "error_mm": z - z_true, **t})
    print(f"{n}: done")

sim_pred = pd.DataFrame(sim_pred_rows)

trial = (sim_pred.groupby(["exposure_ms", "trial"])
         .agg(SNR_dB=("SNR_dB", "mean"), RMSE_mm=("error_mm", lambda e: float(np.sqrt(np.mean(e**2)))),
              bias_mm=("error_mm", "mean"), min_match_score=("min_match_score", "min"))
         .reset_index())
sim_summary = (trial.groupby("exposure_ms")
               .agg(SNR_dB=("SNR_dB", "mean"), RMSE_mean_mm=("RMSE_mm", "mean"),
                    RMSE_CI95_low=("RMSE_mm", lambda x: np.quantile(x, 0.025)),
                    RMSE_CI95_high=("RMSE_mm", lambda x: np.quantile(x, 0.975)),
                    bias_mean_mm=("bias_mm", "mean"), worst_match_score=("min_match_score", "min"))
               .reset_index().sort_values("exposure_ms", ascending=False))
sim_summary["extrapolated"] = sim_summary["exposure_ms"] < min(SIM_EXPOSURES_MS)
display(sim_summary.round(3))

real_1 = avg_summary.loc[avg_summary["frames_averaged"] == 1].iloc[0]
sim_32 = sim_summary.loc[sim_summary["exposure_ms"] == BASE_EXPOSURE_MS].iloc[0]
print(f"Validation at 32 ms single frame:  real SNR {real_1['SNR_dB']:.2f} dB, RMSE {real_1['RMSE_mm']:.2f} mm  |  "
      f"simulated SNR {sim_32['SNR_dB']:.2f} dB, RMSE {sim_32['RMSE_mean_mm']:.2f} mm "
      f"(95% {sim_32['RMSE_CI95_low']:.2f}-{sim_32['RMSE_CI95_high']:.2f})")

lf_holdout_01: done
lf_holdout_02: done


## 2.9 Depth RMSE and Bias vs SNR

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"wspace": 0.3})
interp = sim_summary[~sim_summary["extrapolated"]].sort_values("SNR_dB")
extra = sim_summary[sim_summary["extrapolated"]].sort_values("SNR_dB")

for ax, real_col, sim_col, ylabel, title in [
        (ax1, "RMSE_mm", "RMSE_mean_mm", "Holdout depth RMSE (mm)", "Depth error vs image SNR"),
        (ax2, "bias_mm", "bias_mean_mm", "Mean depth bias (mm)", "Systematic bias vs image SNR")]:
    if ax is ax1:
        ax.fill_between(interp["SNR_dB"], interp["RMSE_CI95_low"], interp["RMSE_CI95_high"], color=ORANGE, alpha=0.18,
                        label="95% Monte Carlo interval")
    ax.plot(interp["SNR_dB"], interp[sim_col], "s-", color=ORANGE, ms=7, label="noise-model injection (16-32 ms)")
    for _, r in interp.iterrows():
        ax.annotate(f"{r['exposure_ms']:.0f} ms", (r["SNR_dB"], r[sim_col]), xytext=(0, -14),
                    textcoords="offset points", ha="center", fontsize=8, color=ORANGE)
    if len(extra):
        ax.plot(extra["SNR_dB"], extra[sim_col], "s:", color=ORANGE, mfc="white", ms=7, label="extrapolated (<16 ms)")
    ax.plot(avg_summary["SNR_dB"], avg_summary[real_col], "o-", color=TEAL, ms=8, label="real multi-frame average")
    for _, r in avg_summary.iterrows():
        ax.annotate(f"{int(r['frames_averaged'])} frame{'s' if r['frames_averaged'] > 1 else ''}", (r["SNR_dB"], r[real_col]),
                    xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8, color=TEAL)
    ax.set(xlabel="Single-view target SNR (dB, linear RAW)", ylabel=ylabel, title=title)
    ax.grid(alpha=0.3)
top = max(avg_summary["RMSE_mm"].max(), sim_summary["RMSE_CI95_high"].max())
ax1.set_ylim(0, top * 1.35)
ax2.set_ylim(min(0, sim_summary["bias_mean_mm"].min()) - 0.5, max(avg_summary["bias_mm"].max(), sim_summary["bias_mean_mm"].max()) * 1.35)
ax2.axhline(0, color=GREY, lw=1, ls="--")
ax1.legend(frameon=False, fontsize=8, loc="lower left")
FIGS["03_depth_rmse_vs_raw_snr"] = fig

## 2.10 Depth RMSE vs SNR (Zoomed, 7.0–7.5 mm)

In [ ]:
YLIM = (7.0, 7.5)
fig, ax = plt.subplots(figsize=(7.2, 4.2))
interp = sim_summary[~sim_summary["extrapolated"]].sort_values("SNR_dB")
ax.fill_between(interp["SNR_dB"], interp["RMSE_CI95_low"], interp["RMSE_CI95_high"], color=ORANGE, alpha=0.18,
                label="95% Monte Carlo interval (noise only)")
ax.plot(interp["SNR_dB"], interp["RMSE_mean_mm"], "s-", color=ORANGE, ms=7, label="noise-model injection (16-32 ms)")
for _, r in interp.iterrows():
    ax.annotate(f"{r['exposure_ms']:.0f} ms", (r["SNR_dB"], r["RMSE_mean_mm"]), xytext=(0, -16),
                textcoords="offset points", ha="center", fontsize=9, color=ORANGE)
ax.plot(avg_summary["SNR_dB"], avg_summary["RMSE_mm"], "o-", color=TEAL, ms=8, label="real multi-frame average")
for _, r in avg_summary.iterrows():
    m = int(r["frames_averaged"])
    ax.annotate(f"{m} frame{'s' if m > 1 else ''}", (r["SNR_dB"], r["RMSE_mm"]), xytext=(0, 10),
                textcoords="offset points", ha="center", fontsize=9, color=TEAL)
means = pd.concat([interp["RMSE_mean_mm"], avg_summary["RMSE_mm"]])
ax.text(0.98, 0.95, f"Mean RMSE varies by only {means.max() - means.min():.2f} mm over "
        f"{interp['SNR_dB'].min():.0f}-{avg_summary['SNR_dB'].max():.0f} dB",
        transform=ax.transAxes, ha="right", va="top", fontsize=10, color=GREY)
ax.set(xlabel="Single-view target SNR (dB, linear RAW)", ylabel="Holdout depth RMSE (mm)",
       title="Depth error vs image SNR (zoomed y-axis)", ylim=YLIM)
ax.grid(alpha=0.3); ax.legend(frameon=False, fontsize=9, loc="lower right")
FIGS["04_depth_rmse_vs_raw_snr_zoomed"] = fig

## 2.11 Save Figures

In [ ]:
saved = []
for name, fig in FIGS.items():
    for ext, kw in (("png", {"dpi": 240}), ("pdf", {})):
        path = FIG_DIR / f"{name}.{ext}"
        fig.savefig(path, bbox_inches="tight", **kw)
        saved.append(path)
print(f"Saved {len(saved)} figure files to {FIG_DIR}:")
for p in saved:
    print(" -", p.name)

## 2.12 Save Tables and Summary

In [ ]:
baseline_snr.to_csv(OUTPUT_DIR / "baseline_single_view_snr.csv", index=False)
measured_snr.to_csv(OUTPUT_DIR / "measured_snr_vs_exposure.csv", index=False)
avg_check.to_csv(OUTPUT_DIR / "sqrtN_averaging_check.csv", index=False)
model_validation.to_csv(OUTPUT_DIR / "noise_model_validation.csv", index=False)
cal_df.to_csv(OUTPUT_DIR / "raw_calibration_slopes.csv", index=False)
avg_pred.to_csv(OUTPUT_DIR / "expA_multiframe_predictions.csv", index=False)
avg_summary.to_csv(OUTPUT_DIR / "expA_multiframe_summary.csv", index=False)
sim_pred.to_csv(OUTPUT_DIR / "expB_noise_injection_predictions.csv", index=False)
sim_summary.to_csv(OUTPUT_DIR / "expB_noise_injection_summary.csv", index=False)

summary = {
    "snr_definition": "linear RAW luminance, (temporal mean - dark) / mean temporal std over repeats, 20*log10",
    "noise_model": {"a": float(NOISE_A), "b": float(NOISE_B)},
    "raw_calibration": {"a_px": float(CAL_A), "b_px_per_mm": float(CAL_B), "R2": float(CAL_R2)},
    "baseline_single_view_snr_dB_mean": float(baseline_snr["SNR_dB"].mean()),
    "expA_multiframe": avg_summary.to_dict(orient="records"),
    "expB_noise_injection": sim_summary.to_dict(orient="records"),
    "noise_model_validation": model_validation.to_dict(orient="records"),
}
with open(OUTPUT_DIR / "raw_snr_depth_summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=float)
print("Saved tables and summary to", OUTPUT_DIR)